In [1]:
!pip install openai-whisper
!pip install python-Levenshtein openpyxl numpy
!apt-get install ffmpeg -y

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=0fc844d98316827ce1065db2998ce57d166b85787181744694e64da586cfa662
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.5 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


In [2]:
import whisper
import os
import subprocess
from openpyxl import load_workbook
from Levenshtein import distance as levenshtein_distance

print("Loading model...")
model = whisper.load_model("medium")  # GPU will be used automatically
print("Model loaded")

Loading model...


100%|█████████████████████████████████████| 1.42G/1.42G [00:36<00:00, 41.4MiB/s]


Model loaded


In [3]:
from google.colab import files
uploaded = files.upload()


Saving 038010_EIT-2A.mp3 to 038010_EIT-2A.mp3
Saving 038011_EIT-1A.mp3 to 038011_EIT-1A.mp3
Saving 038012_EIT-2A.mp3 to 038012_EIT-2A.mp3
Saving 038015_EIT-1A.mp3 to 038015_EIT-1A.mp3
Saving AutoEIT Sample Audio for Transcribing.xlsx to AutoEIT Sample Audio for Transcribing.xlsx


In [4]:
def preprocess_audio(input_path, output_path, skip_seconds=0):
    cmd = [
        "ffmpeg",
        "-y",
        "-ss", str(skip_seconds),
        "-i", input_path,
        "-ar", "16000",
        "-ac", "1",
        output_path
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [5]:
def transcribe_audio(wav_path):
    result = model.transcribe(
        wav_path,
        language="es"
    )

    segments = []
    for seg in result["segments"]:
        segments.append({
            "text": seg["text"].strip(),
            "start": seg["start"],
            "end": seg["end"]
        })

    return segments

In [ ]:
def align_to_stimuli(segments, stimuli, window_size=4):
    aligned = []
    texts = [seg["text"] for seg in segments]

    for stim in stimuli:
        if not stim:
            aligned.append("")
            continue

        best_match = ""
        best_score = float("inf")

        for i in range(len(texts)):
            for j in range(i+1, min(i+window_size, len(texts)) + 1):
                chunk = " ".join(texts[i:j])

                # normalized score
                score = levenshtein_distance(stim.lower(), chunk.lower()) / (len(chunk) + 1)

                if score < best_score:
                    best_score = score
                    best_match = chunk

        aligned.append(best_match)

    return aligned

In [20]:
def process_single(audio_file, stimuli, skip_sec=0):
    wav_file = "temp.wav"

    preprocess_audio(audio_file, wav_file, skip_sec)

    segments = transcribe_audio(wav_file)

    print("Total whisper segments:", len(segments))

    aligned = align_to_stimuli(segments, stimuli)

    results = []
    for stim, trans in zip(stimuli, aligned):
        results.append({
            "stimulus": stim,
            "transcription": trans
        })

    return results

In [22]:
print(wb.sheetnames)

['Info', '38010-2A', '38011-1A', '38012-2A', '38015-1A']


In [ ]:
import re
import os
EXCEL_PATH = "AutoEIT Sample Audio for Transcribing.xlsx"
AUDIO_DIR = "."   # files uploaded directly to root in Colab

def clean_text(text):
    return re.sub(r"\(\d+\)", "", text).strip()

wb = load_workbook(EXCEL_PATH)

print("Available sheets:", wb.sheetnames)

# pick correct participant sheet (skip instructions)
sheet_name = wb.sheetnames[1]
sheet = wb[sheet_name]

print("Using sheet:", sheet_name)


stimuli = [
    clean_text(sheet[f"B{i}"].value) if sheet[f"B{i}"].value else ""
    for i in range(2, 32)
]

print("First 3 stimuli:", stimuli[:3])

print("\nUploaded files:")
print(os.listdir())

audio_file = "038011_EIT-1A.mp3"   # must match EXACT name in uploaded list

# special case (if needed later)
skip_sec = 0
if "038012" in audio_file:
    skip_sec = 720


results = process_single(audio_file, stimuli, skip_sec)


for i in range(5):
    print("\n---")
    print("Stimulus:", results[i]["stimulus"])
    print("Prediction:", results[i]["transcription"])

Available sheets: ['Info', '38010-2A', '38011-1A', '38012-2A', '38015-1A']
Using sheet: 38010-2A
First 3 stimuli: ['Quiero cortarme el pelo', 'El libro está en la mesa', 'El carro lo tiene Pedro']

Uploaded files:
['.config', 'AutoEIT Sample Audio for Transcribing.xlsx', '038011_EIT-1A.mp3', 'temp.wav', '038012_EIT-2A.mp3', '038015_EIT-1A.mp3', '038010_EIT-2A.mp3', 'output.xlsx', 'sample_data']
Total whisper segments: 47

---
Stimulus: Quiero cortarme el pelo
Prediction: Quiero cortarme el pelo.

---
Stimulus: El libro está en la mesa
Prediction: El libro está en la mesa.

---
Stimulus: El carro lo tiene Pedro
Prediction: El carro no tiene pelo.

---
Stimulus: El se ducha cada mañana
Prediction: El se ducha cada mañana.

---
Stimulus: ¿Qué dice usted que va a hacer hoy?
Prediction: ¿Qué dice usted que va a hacer hoy?


In [26]:
for i, res in enumerate(results):
    sheet[f"C{i+2}"] = res["transcription"]

wb.save("output.xlsx")

print("Saved output.xlsx")

Saved output.xlsx


In [18]:
print(results[:5])

[{'stimulus': 'Quiero cortarme el pelo (7)', 'transcription': 'Quiero cortarme el pelo.'}, {'stimulus': 'El libro está en la mesa (7)', 'transcription': 'El libro está en la mesa.'}, {'stimulus': 'El carro lo tiene Pedro (8)', 'transcription': 'El carro no tiene pelo.'}, {'stimulus': 'El se ducha cada mañana (9)', 'transcription': 'El se ducha cada mañana.'}, {'stimulus': '¿Qué dice usted que va a hacer hoy? (9)', 'transcription': '¿Qué dice usted que va a hacer hoy?'}]
